# Payment Transaction Security Scoring

This notebook version of `score_payment_tx_ui.py` lets you score encrypted payment transactions and inspect the output directly in notebook cells.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").exists():
    sys.path.insert(0, str(project_root / "src"))
else:
    sys.path.insert(0, str(project_root))

from crypto_algorithms import ALGORITHM_PROFILES, list_algorithm_profiles, score_encrypted_transaction
from algorithm_benchmark import benchmark_results_as_dicts

list(ALGORITHM_PROFILES.keys())

['RSA-2048', 'ML-KEM-768', 'ML-DSA-65', 'SLH-DSA-128s', 'HQC-128']

## Input Payment Transaction

Change the values below to score a different encrypted transaction payload.

In [2]:
encrypted_payload = "QkFTRTY0X0NJRFBIRVJURVhUX0RFTU9fMTIzNDU2Nzg5MA=="
algorithm = "ML-KEM-768"
amount_sgd = 250000.0
channel = "SWIFT_MT103"

score_result = score_encrypted_transaction(
    encrypted_payload=encrypted_payload,
    algorithm_name=algorithm,
    amount_sgd=amount_sgd,
    channel=channel,
)
score_result

{'algorithm': 'ML-KEM-768',
 'score': 74,
 'risk_rating': 'MEDIUM',
 'quantum_safe': True,
 'entropy': 4.42,
 'valid_encoding': True,
 'findings': ['Ciphertext is unusually short for a protected payment record.',
  'Medium/high-value payment should use PQC or hybrid protection.'],
 'recommendation': 'Keep ML-KEM-768 for key establishment and add ML-DSA-65 for transaction authorization signatures.'}

## Business-Friendly Score View

In [3]:
print(f"Algorithm      : {score_result['algorithm']}")
print(f"Security Score : {score_result['score']}/100")
print(f"Risk Rating    : {score_result['risk_rating']}")
print(f"Quantum Safe   : {score_result['quantum_safe']}")
print(f"Entropy        : {score_result['entropy']}")
print(f"Valid Encoding : {score_result['valid_encoding']}")
print("\nFindings:")
for finding in score_result["findings"]:
    print(f"- {finding}")
print(f"\nRecommendation: {score_result['recommendation']}")

Algorithm      : ML-KEM-768
Security Score : 74/100
Risk Rating    : MEDIUM
Quantum Safe   : True
Entropy        : 4.42
Valid Encoding : True

Findings:
- Ciphertext is unusually short for a protected payment record.
- Medium/high-value payment should use PQC or hybrid protection.

Recommendation: Keep ML-KEM-768 for key establishment and add ML-DSA-65 for transaction authorization signatures.


## Compare Multiple Payment Scenarios

In [4]:
payment_scenarios = [
    {
        "name": "Legacy high-value SWIFT",
        "payload": encrypted_payload,
        "algorithm": "RSA-2048",
        "amount_sgd": 2850000.0,
        "channel": "SWIFT_MT103",
    },
    {
        "name": "PQC protected SWIFT",
        "payload": encrypted_payload,
        "algorithm": "ML-KEM-768",
        "amount_sgd": 2850000.0,
        "channel": "SWIFT_MT103",
    },
    {
        "name": "Signed ACH authorization",
        "payload": encrypted_payload,
        "algorithm": "ML-DSA-65",
        "amount_sgd": 450000.0,
        "channel": "ACH_CREDIT",
    },
]

scenario_rows = []
for scenario in payment_scenarios:
    result = score_encrypted_transaction(
        scenario["payload"],
        scenario["algorithm"],
        scenario["amount_sgd"],
        scenario["channel"],
    )
    scenario_rows.append({
        "scenario": scenario["name"],
        "algorithm": result["algorithm"],
        "score": result["score"],
        "risk_rating": result["risk_rating"],
        "quantum_safe": result["quantum_safe"],
        "recommendation": result["recommendation"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(scenario_rows))
except ImportError:
    for row in scenario_rows:
        print(row)

,scenario,algorithm,score,risk_rating,quantum_safe,recommendation
0,Legacy high-value SWIFT,RSA-2048,0,CRITICAL,False,Migrate payment encryption to hybrid RSA/ECDHE...
1,PQC protected SWIFT,ML-KEM-768,70,MEDIUM,True,Keep ML-KEM-768 for key establishment and add ...
2,Signed ACH authorization,ML-DSA-65,72,MEDIUM,True,Use ML-DSA-65 for signatures and ML-KEM-768 fo...


## Algorithm Scorecard

In [5]:
try:
    profiles_df = pd.DataFrame(list_algorithm_profiles())
    display(profiles_df[[
        "name",
        "primary_use",
        "nist_status",
        "quantum_safe",
        "security_level",
        "base_score",
    ]])
except NameError:
    for profile in list_algorithm_profiles():
        print(profile)

,name,primary_use,nist_status,quantum_safe,security_level,base_score
0,RSA-2048,Encryption/signature,Classical legacy,False,Classical ~112-bit; vulnerable to Shor's algor...,46
1,ML-KEM-768,Key encapsulation/encryption,FIPS 203,True,NIST category 3,93
2,ML-DSA-65,Digital signature,FIPS 204,True,NIST category 3,91
3,SLH-DSA-128s,Digital signature,FIPS 205,True,NIST category 1,84
4,HQC-128,Key encapsulation/encryption,Selected by NIST in 2025 for future standardiz...,True,NIST category 1 target,82


## Optional: Benchmark From the Same Notebook

In [6]:
benchmark_rows = benchmark_results_as_dicts(iterations=3)
try:
    display(pd.DataFrame(benchmark_rows))
except NameError:
    for row in benchmark_rows:
        print(row)

,algorithm,primitive,implementation,avg_ms,p95_ms,security_score,quantum_safe,notes
0,RSA-2048,Encryption + signature,cryptography real RSA,104.807,160.039,46,False,"Legacy baseline: real RSA-OAEP key wrap, AES-G..."
1,ML-KEM-768,KEM + AEAD encryption,simulation fallback,0.179,0.407,93,True,Primary PQC encryption/KEM path for protecting...
2,ML-DSA-65,Digital signature,simulation fallback,0.114,0.124,91,True,Primary PQC transaction authorization signature.
3,SLH-DSA-128s,Digital signature,simulation fallback,0.220,0.244,84,True,Hash-based backup signature; useful for crypto...
4,HQC-128,KEM + AEAD encryption,simulation fallback,0.017,0.028,82,True,Code-based backup KEM candidate with different...
